# ICU Length of Stay Prediction

In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import polars as pl

from ethos.constants import PROJECT_ROOT
from ethos.constants import SpecialToken as ST
from ethos.inference.constants import Reason, Task
from ethos.metrics import compute_and_print_metrics_regression, preprocess_inference_results
from ethos.utils import get_mimic_hf_patients, get_mimic_sepsis_icu_stays

# Requires original MIMIC-IV 2.2 dataset that we do not provide
mimic_sepsis = get_mimic_sepsis_icu_stays()
# heart_failure_cohort = get_mimic_hf_patients(PROJECT_ROOT / "data/mimic-2.2")

/home/henri/ethos-benchmark/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# DEBUGGING ONLY
import sys
from pathlib import Path

project_root = Path.cwd()
while not (project_root / 'pyproject.toml').exists() and project_root != project_root.parent:
    project_root = project_root.parent

sys.path.insert(0, str(project_root))
from src.ethos.datasets.length_of_stay import LosPredictionDataset
path = "../../../../mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test"
dataset = LosPredictionDataset(path)

Found 10997 hospital stays with an outcome token.


In [3]:
import pandas as pd
df_token = pd.read_parquet("/mnt/data_share/project_henri/ethos-ares/mimic-tokenized/test/11_inject_time_intervals/0.parquet")

# MIMIC

In [ ]:
results_dir = PROJECT_ROOT / "results"
los_pred_result_dir = results_dir / Task.LOS_PREDICTION

def preprocess_los_prediction_results(input_dir) -> pl.DataFrame:
    return preprocess_inference_results(
        input_dir,
        actual_expr=pl.col("actual") == ST.DEATH,
        expected_expr=pl.col("expected") == ST.DEATH,
        filter_ambiguous=(
            pl.col("actual").is_in([ST.DEATH, ST.ICU_DISCHARGE])
            & (pl.col("stop_reason") == Reason.GOT_TOKEN)
        ),
    )

## Hospital Mortality

In [ ]:
hospital_mortality_results = []
for result_fp in los_pred_result_dir.iterdir():
    df = preprocess_los_prediction_results(result_fp)
    rep_num = df["counts"].mean()
    plt.figure(figsize=(10, 5))
    plt.suptitle(f"Hospital Mortality\n{result_fp.name} (rep_num={rep_num:.2f})")
    plt.subplot(1, 2, 1)
    res_full = compute_and_print_metrics_regression(df["expected"], df["actual"], "Full")

    plt.subplot(1, 2, 2)
    # hf_df = df.filter(pl.col("patient_id").is_in(heart_failure_cohort))
    # res_hf = compute_and_print_metrics(hf_df["expected"], hf_df["actual"], "HF patients")

    hospital_mortality_results.append(
        {
            "name": result_fp.name,
            "auc": res_full["fitted_auc"],
            # "auc_hf": res_hf["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.tight_layout()
    plt.show()
hospital_mortality_results = pl.from_dicts(hospital_mortality_results).sort("auc", descending=True)

FileNotFoundError: [Errno 2] No such file or directory: '/home/henri/ethos-benchmark/results/hospital_mortality'

In [ ]:
hospital_mortality_results

## ICU Mortality

In [ ]:
icu_mortality_results = []
for result_fp in icu_mort_result_dir.iterdir():
    df = preprocess_los_prediction_results(result_fp)
    rep_num = df["counts"].mean()
    plt.figure(figsize=(10, 5))
    plt.suptitle(f"ICU Mortality\n{result_fp.name} (rep_num={rep_num:.2f})")
    plt.subplot(1, 2, 1)
    res_full = compute_and_print_metrics(df["expected"], df["actual"], "Full")

    plt.subplot(1, 2, 2)
    # hf_df = df.filter(pl.col("patient_id").is_in(heart_failure_cohort))
    # res_hf = compute_and_print_metrics(hf_df["expected"], hf_df["actual"], "HF patients")

    icu_mortality_results.append(
        {
            "name": result_fp.name,
            "auc": res_full["fitted_auc"],
            # "auc_hf": res_hf["fitted_auc"],
            "rep_num": rep_num,
        }
    )
    plt.tight_layout()
    plt.show()
icu_mortality_results = pl.from_dicts(icu_mortality_results).sort("auc", descending=True)

In [ ]:
icu_mortality_results